In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from src.utils import load_and_vectorize_data
from src.transformer_model import Transformer

In [3]:
BATCH_SIZE = 32
D_MODEL = 128
NUM_HEADS = 4
D_FF = 512
NUM_LAYERS = 3
EPOCHS = 10
LEARNING_RATE = 0.0005

In [4]:
enc_in, dec_in, dec_tar, src_vec, trg_vec, max_src, max_trg = load_and_vectorize_data()

Vectorization Ready. English Vocab: 3355, French Vocab: 7801
Max Source Length: 5, Max Target Length: 12


In [5]:
dataset = TensorDataset(enc_in, dec_in, dec_tar)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [6]:
model = Transformer(
    src_vocab_size=src_vec.vocab_size,
    trg_vocab_size=trg_vec.vocab_size,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS
)

In [7]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [8]:
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    
    for batch_idx, (src, dec_i, dec_t) in enumerate(dataloader):
        optimizer.zero_grad()
        
        # outputs shape: (batch_size, seq_len, trg_vocab_size)
        outputs = model(src, dec_i)
        
        # Flatten target and prediction tensors for CrossEntropy
        loss = criterion(
            outputs.view(-1, trg_vec.vocab_size), 
            dec_t.view(-1)
        )
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] complete. Average Loss: {avg_loss:.4f}")

Epoch [1/10] complete. Average Loss: 4.2288
Epoch [2/10] complete. Average Loss: 2.7432
Epoch [3/10] complete. Average Loss: 2.0082
Epoch [4/10] complete. Average Loss: 1.4899
Epoch [5/10] complete. Average Loss: 1.1228
Epoch [6/10] complete. Average Loss: 0.8729
Epoch [7/10] complete. Average Loss: 0.7074
Epoch [8/10] complete. Average Loss: 0.6176
Epoch [9/10] complete. Average Loss: 0.5596
Epoch [10/10] complete. Average Loss: 0.5266


In [11]:
def translate_sentence(sentence, model, src_vectorizer, trg_vectorizer, max_trg_len):
    model.eval() # Set model to evaluation mode
    
    # 1. PREPROCESS SOURCE (ENGLISH) TEXT
    # Vectorize the raw string input using your trained TextVectorization layer
    # Add a batch dimension using unsqueeze(0) -> shape: (1, max_src_len)
    src_tokens = src_vectorizer([sentence])
    src_tensor = torch.tensor(src_tokens.numpy(), dtype=torch.long)
    
    # 2. INITIALIZE DECODER WITH START TOKEN
    # Look up the index for "startseq" dynamically from your vocabulary mapping
    start_idx = trg_vectorizer.word_to_index["startseq"]
    end_idx = trg_vectorizer.word_to_index["endseq"]
    
    # Initialize our running tracker of predicted target tokens
    # Start shape: (1, 1) -> contains just [[start_idx]]
    trg_indices = [[start_idx]]
    
    # 3. AUTOREGRESSIVE GENERATION LOOP
    with torch.no_grad(): # Disable gradient calculations to save memory
        for _ in range(max_trg_len):
            # Convert current list of target tokens into a PyTorch tensor
            trg_tensor = torch.tensor(trg_indices, dtype=torch.long)
            
            # Forward pass: pass the entire source sequence and the targets generated SO FAR
            # Output shape: (1, current_seq_len, trg_vocab_size)
            outputs = model(src_tensor, trg_tensor)
            
            # Grab the prediction for the VERY LAST token position only
            # shape: (trg_vocab_size,)
            next_token_logits = outputs[0, -1, :]
            
            # Pick the word index with the highest probability
            next_token_id = torch.argmax(next_token_logits).item()
            
            # Append the predicted word ID to our decoder sequence tracker
            trg_indices[0].append(next_token_id)
            
            # Stop generating immediately if the model predicts the end-of-sentence token
            if next_token_id == end_idx:
                break
                
    # 4. CONVERT TOKEN IDS BACK TO STRINGS
    vocab = trg_vectorizer.get_vocabulary()
    translated_words = [vocab[idx] for idx in trg_indices[0] if vocab[idx] not in ["startseq", "endseq", ""]]
    
    return " ".join(translated_words)

In [28]:
test_sentence = "I am honest"
translation = translate_sentence(
    sentence=test_sentence,
    model=model,
    src_vectorizer=src_vec,
    trg_vectorizer=trg_vec,
    max_trg_len=max_trg,
)

print(f"English: {test_sentence}")
print(f"French Translation: {translation}")

English: I am honest
French Translation: je suis honnête
